<a href="https://colab.research.google.com/github/AnirudhSrinivasan77/ai-output-safety-classifier/blob/main/MEGAPROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import requests
import zipfile
import pandas as pd
import json

base_dir = "/content/AI_Output_Safety_Classifier"
raw_dir = os.path.join(base_dir, "data/raw")
processed_dir = os.path.join(base_dir, "data/processed")

# Only 4 datasets now
datasets = ["jigsaw_toxic", "liar", "hate_speech", "truthfulqa"]

# Create folders
for d in datasets:
    os.makedirs(os.path.join(raw_dir, d), exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)
print("✅ Folder structure ready.")

# === Kaggle setup for Jigsaw Toxic ===
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json here

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle credentials set.")

# === Download datasets ===
# 1. Jigsaw Toxic - Kaggle
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge -p /content/AI_Output_Safety_Classifier/data/raw/jigsaw_toxic

# 2. LIAR dataset (CSV)
liar_url = "https://raw.githubusercontent.com/Tariq60/LIAR-Dataset-Cleaned/main/liar.csv"
liar_dest = f"{raw_dir}/liar/liar.csv"
response = requests.get(liar_url)
with open(liar_dest, 'wb') as f:
    f.write(response.content)
print(f"✅ Downloaded LIAR dataset to: {liar_dest}")

# 3. Hate Speech (ZIP)
hate_url = "https://github.com/t-davidson/hate-speech-and-offensive-language/archive/refs/heads/master.zip"
hate_zip_path = f"{raw_dir}/hate_speech/hate.zip"
response = requests.get(hate_url)
with open(hate_zip_path, 'wb') as f:
    f.write(response.content)
print(f"✅ Downloaded Hate Speech dataset to: {hate_zip_path}")

# 4. TruthfulQA (ZIP)
truthfulqa_url = "https://github.com/sylinrl/TruthfulQA/archive/refs/heads/main.zip"
truthfulqa_zip_path = f"{raw_dir}/truthfulqa/truthfulqa.zip"
response = requests.get(truthfulqa_url)
with open(truthfulqa_zip_path, 'wb') as f:
    f.write(response.content)
print(f"✅ Downloaded TruthfulQA dataset to: {truthfulqa_zip_path}")

# === Extract zips ===
def unzip_file(zip_path, extract_to):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
            print(f"✅ Extracted: {zip_path}")
    except zipfile.BadZipFile:
        print(f"❌ Bad zip: {zip_path}")

# Extract Jigsaw Toxic (all zip files in folder)
jigsaw_folder = os.path.join(raw_dir, "jigsaw_toxic")
for file in os.listdir(jigsaw_folder):
    if file.endswith(".zip"):
        unzip_file(os.path.join(jigsaw_folder, file), jigsaw_folder)

# Extract hate_speech and truthfulqa
unzip_file(hate_zip_path, f"{raw_dir}/hate_speech")
unzip_file(truthfulqa_zip_path, f"{raw_dir}/truthfulqa")

# === Process datasets ===
processed_rows = []

import zipfile
import os

folder = "/content/AI_Output_Safety_Classifier/data/raw/jigsaw_toxic"
zip_files = [f for f in os.listdir(folder) if f.endswith('.zip')]

for zfile in zip_files:
    zip_path = os.path.join(folder, zfile)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(folder)
    print(f"✅ Extracted {zfile}")

# 1. Jigsaw Toxic
jigsaw_path = os.path.join(raw_dir, "jigsaw_toxic", "train.csv")  # Check exact file after extraction
df_jigsaw = pd.read_csv(jigsaw_path)
for _, row in df_jigsaw.iterrows():
    processed_rows.append({"text": row["comment_text"], "label": int(row["toxic"])})

# 2. LIAR
liar_path = os.path.join(raw_dir, "liar", "liar.csv")
df_liar = pd.read_csv(liar_path)
safe_labels = ['true', 'mostly-true', 'half-true']
for _, row in df_liar.iterrows():
    label = 0 if row["label"].strip().lower() in safe_labels else 1
    processed_rows.append({"text": row["statement"], "label": label})

# 3. Hate Speech
# The extracted folder is hate-speech-and-offensive-language-master/
hs_path = os.path.join(raw_dir, "hate_speech", "hate-speech-and-offensive-language-master", "data", "labeled_data.csv")
df_hate = pd.read_csv(hs_path)
for _, row in df_hate.iterrows():
    label = 1 if row["class"] in [0, 1] else 0
    processed_rows.append({"text": row["tweet"], "label": label})

# 4. TruthfulQA
truth_path = os.path.join(raw_dir, "truthfulqa", "TruthfulQA-main", "data", "mc_task.json")
with open(truth_path, "r") as f:
    qa_data = json.load(f)
for q in qa_data:
    processed_rows.append({"text": q["question"], "label": 0})  # All considered safe

# === Save combined CSV ===
combined_df = pd.DataFrame(processed_rows)
out_path = os.path.join(processed_dir, "combined_dataset.csv")
combined_df.to_csv(out_path, index=False)

print(f"✅ Final dataset shape: {combined_df.shape}")
print(f"📁 Saved combined dataset at: {out_path}")

✅ Folder structure ready.


Saving kaggle.json to kaggle.json
✅ Kaggle credentials set.
  0% 0.00/52.6M [00:00<?, ?B/s]
100% 52.6M/52.6M [00:00<00:00, 1.32GB/s]
✅ Downloaded LIAR dataset to: /content/AI_Output_Safety_Classifier/data/raw/liar/liar.csv
✅ Downloaded Hate Speech dataset to: /content/AI_Output_Safety_Classifier/data/raw/hate_speech/hate.zip
✅ Downloaded TruthfulQA dataset to: /content/AI_Output_Safety_Classifier/data/raw/truthfulqa/truthfulqa.zip
✅ Extracted: /content/AI_Output_Safety_Classifier/data/raw/jigsaw_toxic/jigsaw-toxic-comment-classification-challenge.zip
✅ Extracted: /content/AI_Output_Safety_Classifier/data/raw/hate_speech/hate.zip
✅ Extracted: /content/AI_Output_Safety_Classifier/data/raw/truthfulqa/truthfulqa.zip
✅ Extracted test.csv.zip
✅ Extracted sample_submission.csv.zip
✅ Extracted test_labels.csv.zip
✅ Extracted train.csv.zip
✅ Extracted jigsaw-toxic-comment-classification-challenge.zip
✅ Final dataset shape: (185144, 2)
📁 Saved combined dataset at: /content/AI_Output_Safety_Class

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

data_path = "/content/AI_Output_Safety_Classifier/data/processed/combined_dataset.csv"
df = pd.read_csv(data_path)

print(df.head())
print(df.label.value_counts())

# Split into train and test (e.g., 80-20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

                                                text  label
0  Explanation\nWhy the edits made under my usern...      0
1  D'aww! He matches this background colour I'm s...      0
2  Hey man, I'm really not trying to edit war. It...      0
3  "\nMore\nI can't make any real suggestions on ...      0
4  You, sir, are my hero. Any chance you remember...      0
label
0    149230
1     35914
Name: count, dtype: int64
Train size: 148115, Test size: 37029


In [ ]:
!pip install transformers torch


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

# 1. Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 2. Custom Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),      # Flatten to 1D tensor
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 3. Create Dataset objects
train_dataset = TextDataset(train_df['text'], train_df['label'], tokenizer)
test_dataset = TextDataset(test_df['text'], test_df['label'], tokenizer)

# 4. Create DataLoaders
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Train batches: 9258
Test batches: 2315


In [ ]:
import torch.nn as nn
from transformers import BertModel

class BertClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(self.bert.config.hidden_size, 2)  # 2 classes

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [batch_size, hidden_size]
        dropped = self.dropout(pooled_output)
        return self.linear(dropped)


In [ ]:
import torch
from torch import optim
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_epoch(model, data_loader, optimizer, criterion):
    model.train()
    losses = []
    all_preds = []
    all_labels = []

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return sum(losses)/len(losses), acc

def eval_model(model, data_loader, criterion):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            losses.append(loss.item())
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return sum(losses)/len(losses), acc


In [ ]:
from tqdm import tqdm
import torch
from sklearn.metrics import accuracy_score

# Update train_epoch with progress bar
def train_epoch(model, data_loader, optimizer, criterion):
    model.train()
    losses, all_preds, all_labels = [], [], []

    loop = tqdm(data_loader, desc="Training", leave=False)
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        loop.set_postfix(loss=loss.item())

    acc = accuracy_score(all_labels, all_preds)
    return sum(losses) / len(losses), acc


def eval_model(model, data_loader, criterion):
    model.eval()
    losses, all_preds, all_labels = [], [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            losses.append(loss.item())
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return sum(losses) / len(losses), acc


# Use smaller sample to test speed
train_df_small = train_df.sample(5000, random_state=42)
test_df_small = test_df.sample(1000, random_state=42)

train_dataset = TextDataset(train_df_small['text'], train_df_small['label'], tokenizer)
test_dataset = TextDataset(test_df_small['text'], test_df_small['label'], tokenizer)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16)

# Initialize model, optimizer, criterion
model = BertClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

epochs = 3
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_model(model, test_loader, criterion)

    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1/3
Train Loss: 0.1944, Train Acc: 0.9282
Val Loss: 0.1056, Val Acc: 0.9680


Epoch 2/3
Train Loss: 0.0830, Train Acc: 0.9722
Val Loss: 0.0937, Val Acc: 0.9640


Epoch 3/3
Train Loss: 0.0376, Train Acc: 0.9880
Val Loss: 0.1438, Val Acc: 0.9630


In [ ]:
import torch
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np
import os

# Make sure Drive is mounted
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
from transformers import BertModel

# Redefine architecture (must match your original model)
class BertClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        dropped = self.dropout(pooled_output)
        return self.linear(dropped)

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertClassifier()
model.load_state_dict(torch.load("/content/drive/MyDrive/MainProject/bert_classifier_best.pt", map_location=device))
model = model.to(device)
model.eval()

print("✅ Model structure verified and successfully loaded!")


from transformers import BertTokenizer
import torch

# Paths
model_path = "/content/drive/MyDrive/MainProject/bert_classifier_best.pt"
tokenizer_path = "/content/drive/MyDrive/MainProject/tokenizer"

# Load trained model (already done above)
model.eval()

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained(tokenizer_path)
print(f"✅ Tokenizer loaded from {tokenizer_path}")

# Inference function
def predict(text, model, tokenizer, device, max_len=128):
    model.eval()
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        probs = torch.softmax(outputs, dim=1)
        pred = torch.argmax(probs, dim=1).cpu().item()

    return "Unsafe" if pred == 1 else "Safe"

# Example usage
sample_text = "You are an boy!"
result = predict(sample_text, model, tokenizer, device)
print(f"Prediction: {result}")



Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✅ Model structure verified and successfully loaded!
✅ Tokenizer loaded from /content/drive/MyDrive/MainProject/tokenizer
Prediction: Safe
